# Swap sensitivity check: accuracy before vs. after randomizing V0/V1 orientation

Notebook 03 randomly swaps each pair's V0/V1 orientation (with probability 0.5) before any
model is trained, so that a model cannot win by exploiting the position in the file order
instead of the code itself. In the raw dataset the second submission happens to use less
energy on average (mean 640 J vs. 837 J), so a model trained on the **un-swapped** data could
look artificially accurate by learning "the version listed second tends to win" rather than
anything about the code.

Per supervisor feedback, this notebook measures predictive quality **before** swapping
(original V0/V1 order, as released in the raw dataset) and **after** swapping (the current,
randomized order used everywhere else in this project) for the same models and the same
problem-grouped cross-validation. A large accuracy drop from "before" to "after" would indicate
that the un-swapped accuracy was partly inflated by positional shortcut learning; a small or no
drop indicates the model is relying on genuine code signal.

Run Notebook 03 first. This notebook only reads the already-prepared parquet files; it does not
change any file written by Notebook 03 or 04.

In [3]:
import json
from pathlib import Path

import numpy as np
import pandas as pd
from IPython.display import display
from sklearn.dummy import DummyClassifier
from sklearn.ensemble import RandomForestClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score, balanced_accuracy_score, f1_score, roc_auc_score
from sklearn.model_selection import GroupKFold
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import StandardScaler
from xgboost import XGBClassifier

ROOT = next(
    (path for path in (Path.cwd(), *Path.cwd().parents) if (path / "Data" / "Processed").is_dir()),
    None,
)
if ROOT is None:
    raise FileNotFoundError("Run 03_feature_engineering.ipynb first to create Data/Processed.")
PROCESSED_DIR = ROOT / "Data" / "Processed"
RESULTS_DIR = ROOT / "Results" / "swap_sensitivity"
RESULTS_DIR.mkdir(parents=True, exist_ok=True)

manifest = json.loads((PROCESSED_DIR / "feature_manifest.json").read_text(encoding="utf-8"))
feature_configurations = {
    "config_A": manifest["configurations"]["per_version"],
    "config_B": manifest["configurations"]["differences"],
    "combined": manifest["configurations"]["combined"],
}

datasets_swapped = {
    "validated": pd.read_parquet(PROCESSED_DIR / "prepared_training_dataset.parquet"),
    "significant": pd.read_parquet(PROCESSED_DIR / "prepared_significant_training_dataset.parquet"),
}
for name, frame in datasets_swapped.items():
    if "was_swapped" not in frame.columns:
        raise ValueError(f"{name}: expected a 'was_swapped' column written by Notebook 03.")
    print(f"{name}: {len(frame):,} pairs, {int(frame['was_swapped'].sum()):,} were swapped.")

validated: 21,410 pairs, 10,611 were swapped.
significant: 3,000 pairs, 1,514 were swapped.


### Reconstruct the un-swapped (original V0/V1 order) dataset

Swapping is its own inverse: applying the same V0/V1 swap a second time to exactly the rows
that were swapped the first time restores the original raw order. This avoids re-running
Notebook 01's cleaning and Notebook 03's feature extraction from scratch; it only inverts the
orientation transformation on the already-prepared, already-feature-engineered frame.

In [4]:
def invert_swap(frame):
    """Return a copy of frame with V0/V1 orientation restored to the original raw order."""
    frame = frame.copy()
    swap_mask = frame["was_swapped"].astype(bool)
    if not swap_mask.any():
        return frame

    # Columns such as "energy_difference_v1_minus_v0" end in "_v0" purely by
    # coincidence of their name (the "...minus_v0" part) -- they are a single
    # precomputed directional difference, not a genuine per-version V0/V1
    # feature pair, and must be sign-negated rather than value-swapped.
    directional_difference_columns = [
        column for column in frame.columns if column.endswith("_v1_minus_v0")
    ]
    directional_difference_column_set = set(directional_difference_columns)

    v0_bases = {
        column[:-3]
        for column in frame.columns
        if column.endswith("_v0") and column not in directional_difference_column_set
    }
    v1_bases = {
        column[:-3]
        for column in frame.columns
        if column.endswith("_v1") and column not in directional_difference_column_set
    }
    if v0_bases != v1_bases:
        raise ValueError(
            f"Unpaired version columns found: V0-only={sorted(v0_bases - v1_bases)}, "
            f"V1-only={sorted(v1_bases - v0_bases)}."
        )
    for base in sorted(v0_bases):
        column_v0, column_v1 = f"{base}_v0", f"{base}_v1"
        old_v0 = frame.loc[swap_mask, column_v0].copy()
        frame.loc[swap_mask, column_v0] = frame.loc[swap_mask, column_v1].to_numpy()
        frame.loc[swap_mask, column_v1] = old_v0.to_numpy()
    # Note: this loop already swaps code_v0_no_empty_lines/code_v1_no_empty_lines
    # (base "code_no_empty_lines") -- no separate special case needed for those.

    delta_columns = [column for column in frame.columns if column.endswith("_delta")]
    for column in delta_columns:
        frame.loc[swap_mask, column] = -frame.loc[swap_mask, column]

    for column in directional_difference_columns:
        frame.loc[swap_mask, column] = -frame.loc[swap_mask, column]

    # "code_v0_no_empty_lines"/"code_v1_no_empty_lines" do not end in "_v0"/"_v1"
    # (they end in "_no_empty_lines", with "_v0"/"_v1" in the middle), so the
    # generic base-pairing loop above never touches them -- they need their own
    # explicit swap.
    if "code_v0_no_empty_lines" in frame.columns and "code_v1_no_empty_lines" in frame.columns:
        old_code_v0 = frame.loc[swap_mask, "code_v0_no_empty_lines"].copy()
        frame.loc[swap_mask, "code_v0_no_empty_lines"] = (
            frame.loc[swap_mask, "code_v1_no_empty_lines"].to_numpy()
        )
        frame.loc[swap_mask, "code_v1_no_empty_lines"] = old_code_v0.to_numpy()

    if "energy_winner" in frame.columns:
        frame.loc[swap_mask, "energy_winner"] = 1 - frame.loc[swap_mask, "energy_winner"]
    frame["was_swapped"] = False
    return frame


datasets_unswapped = {name: invert_swap(frame) for name, frame in datasets_swapped.items()}
for name in datasets_swapped:
    recovered_mean_v1 = datasets_unswapped[name]["energy_winner"].mean()
    print(f"{name} (original order): share of pairs where V1 wins = {recovered_mean_v1:.3f}")
    swapped_mean_v1 = datasets_swapped[name]["energy_winner"].mean()
    print(f"{name} (after swap, current):  share of pairs where V1 wins = {swapped_mean_v1:.3f}")

validated (original order): share of pairs where V1 wins = 0.593
validated (after swap, current):  share of pairs where V1 wins = 0.499
significant (original order): share of pairs where V1 wins = 0.779
significant (after swap, current):  share of pairs where V1 wins = 0.507


### Train and evaluate both conditions with identical cross-validation

Config A (per-version features) is the most diagnostic configuration here: it is the one
configuration where a model could latch onto "whichever version appears second" as a shortcut
in the original, un-swapped order. Config B (differences) and the combined configuration are
included for completeness, since the thesis reports all three elsewhere.

In [5]:
model_factories = {
    "majority_baseline": lambda: DummyClassifier(strategy="most_frequent"),
    "logistic_regression": lambda: make_pipeline(
        StandardScaler(),
        LogisticRegression(max_iter=2000, random_state=42),
    ),
    "random_forest": lambda: RandomForestClassifier(
        n_estimators=200, max_depth=12, random_state=42, n_jobs=-1,
        class_weight="balanced_subsample",
    ),
    "xgboost": lambda: XGBClassifier(
        n_estimators=150, max_depth=6, learning_rate=0.1, random_state=42,
        n_jobs=4, eval_metric="logloss",
    ),
}


def evaluate_condition(frame, columns, model_name, factory):
    features = frame[columns].astype(float)
    target = frame["energy_winner"].astype(int)
    groups = frame["group_id"]
    rows = []
    for fold, (train_idx, valid_idx) in enumerate(
        GroupKFold(n_splits=5).split(features, target, groups), start=1
    ):
        model = factory()
        model.fit(features.iloc[train_idx], target.iloc[train_idx])
        truth = target.iloc[valid_idx]
        prediction = model.predict(features.iloc[valid_idx]).astype(int)
        try:
            probability = model.predict_proba(features.iloc[valid_idx])[:, 1]
            roc_auc = roc_auc_score(truth, probability) if truth.nunique() == 2 else np.nan
        except (AttributeError, IndexError):
            roc_auc = np.nan
        rows.append({
            "fold": fold,
            "accuracy": accuracy_score(truth, prediction),
            "balanced_accuracy": balanced_accuracy_score(truth, prediction),
            "roc_auc": roc_auc,
            "f1": f1_score(truth, prediction, zero_division=0),
        })
    fold_frame = pd.DataFrame(rows)
    return {
        "accuracy_mean": fold_frame["accuracy"].mean(),
        "balanced_accuracy_mean": fold_frame["balanced_accuracy"].mean(),
        "roc_auc_mean": fold_frame["roc_auc"].mean(),
        "f1_mean": fold_frame["f1"].mean(),
    }


conditions = {
    "before_swap_original_order": datasets_unswapped,
    "after_swap_randomized_order": datasets_swapped,
}

records = []
for condition_name, frames_by_dataset in conditions.items():
    for dataset_name, frame in frames_by_dataset.items():
        for config_name, columns in feature_configurations.items():
            for model_name, factory in model_factories.items():
                metrics = evaluate_condition(frame, columns, model_name, factory)
                records.append({
                    "condition": condition_name,
                    "dataset": dataset_name,
                    "feature_config": config_name,
                    "model": model_name,
                    **metrics,
                })
                print(
                    f"{condition_name} | {dataset_name} | {config_name} | {model_name}: "
                    f"accuracy={metrics['accuracy_mean']:.3f}, roc_auc={metrics['roc_auc_mean']:.3f}"
                )

swap_sensitivity = pd.DataFrame(records)
swap_sensitivity.to_csv(RESULTS_DIR / "swap_sensitivity_results.csv", index=False)
display(swap_sensitivity)

before_swap_original_order | validated | config_A | majority_baseline: accuracy=0.593, roc_auc=0.500
before_swap_original_order | validated | config_A | logistic_regression: accuracy=0.625, roc_auc=0.639
before_swap_original_order | validated | config_A | random_forest: accuracy=0.582, roc_auc=0.621
before_swap_original_order | validated | config_A | xgboost: accuracy=0.613, roc_auc=0.621
before_swap_original_order | validated | config_B | majority_baseline: accuracy=0.593, roc_auc=0.500
before_swap_original_order | validated | config_B | logistic_regression: accuracy=0.625, roc_auc=0.635
before_swap_original_order | validated | config_B | random_forest: accuracy=0.557, roc_auc=0.630
before_swap_original_order | validated | config_B | xgboost: accuracy=0.619, roc_auc=0.623
before_swap_original_order | validated | combined | majority_baseline: accuracy=0.593, roc_auc=0.500
before_swap_original_order | validated | combined | logistic_regression: accuracy=0.625, roc_auc=0.640
before_swap_

,condition,dataset,feature_config,model,accuracy_mean,balanced_accuracy_mean,roc_auc_mean,f1_mean
0,before_swap_original_order,validated,config_A,majority_baseline,0.593087,0.500000,0.500000,0.744562
1,before_swap_original_order,validated,config_A,logistic_regression,0.624942,0.560726,0.639444,0.741279
2,before_swap_original_order,validated,config_A,random_forest,0.582438,0.576432,0.620994,0.633416
3,before_swap_original_order,validated,config_A,xgboost,0.612985,0.560371,0.620509,0.720826
4,before_swap_original_order,validated,config_B,majority_baseline,0.593087,0.500000,0.500000,0.744562
5,before_swap_original_order,validated,config_B,logistic_regression,0.624521,0.558560,0.634562,0.742563
6,before_swap_original_order,validated,config_B,random_forest,0.557450,0.584550,0.630062,0.541048
7,before_swap_original_order,validated,config_B,xgboost,0.619010,0.556123,0.622744,0.735648
8,before_swap_original_order,validated,combined,majority_baseline,0.593087,0.500000,0.500000,0.744562
9,before_swap_original_order,validated,combined,logistic_regression,0.624988,0.561009,0.639625,0.741084


### Before vs. after comparison table

In [6]:
comparison = swap_sensitivity.pivot_table(
    index=["dataset", "feature_config", "model"],
    columns="condition",
    values=["accuracy_mean", "roc_auc_mean"],
)
comparison["accuracy_gap"] = (
    comparison[("accuracy_mean", "before_swap_original_order")]
    - comparison[("accuracy_mean", "after_swap_randomized_order")]
)
comparison["roc_auc_gap"] = (
    comparison[("roc_auc_mean", "before_swap_original_order")]
    - comparison[("roc_auc_mean", "after_swap_randomized_order")]
)
comparison = comparison.sort_values("accuracy_gap", ascending=False)
comparison.to_csv(RESULTS_DIR / "swap_sensitivity_comparison.csv")
display(comparison)

print(
    "\nA positive gap means the model looked more accurate on the original, un-swapped order "
    "than it does once position is randomized -- i.e. part of its apparent accuracy came from "
    "position rather than code. A gap near zero across Config A is the result that supports "
    "using this model's static-feature accuracy at face value."
)

C:\Users\User\AppData\Local\Temp\ipykernel_37632\434851550.py:6: Pandas4Warning: Setting a new column on a DataFrame with a MultiIndex using a key that is not a full-length tuple is deprecated and will raise in a future version. Use a full-length tuple key instead.
  comparison["accuracy_gap"] = (
C:\Users\User\AppData\Local\Temp\ipykernel_37632\434851550.py:10: Pandas4Warning: Setting a new column on a DataFrame with a MultiIndex using a key that is not a full-length tuple is deprecated and will raise in a future version. Use a full-length tuple key instead.
  comparison["roc_auc_gap"] = (


accuracy_mean  \
condition                                      after_swap_randomized_order   
dataset     feature_config model                                             
significant combined       majority_baseline                      0.507000   
            config_A       majority_baseline                      0.507000   
            config_B       majority_baseline                      0.507000   
            config_A       random_forest                          0.641333   
                           xgboost                                0.648667   
            config_B       xgboost                                0.652333   
            combined       random_forest                          0.660000   
            config_A       logistic_regression                    0.657000   
            combined       xgboost                                0.657667   
                           logistic_regression                    0.659333   
            config_B       logistic_regression                    0.662667   
                           random_forest                          0.657333   
validated   combined       majority_baseline                      0.498225   
            config_A       majority_baseline                      0.498225   
            config_B       majority_baseline                      0.498225   
            config_A       xgboost                                0.579169   
            config_B       xgboost                                0.587716   
            combined       logistic_regression                    0.594769   
                           xgboost                                0.585194   
            config_A       logistic_regression                    0.596217   
            config_B       logistic_regression                    0.598412   
            config_A       random_forest                          0.583652   
            combined       random_forest                          0.589351   
            config_B       random_forest                          0.594535   

                                                                           \
condition                                      before_swap_original_order   
dataset     feature_config model                                            
significant combined       majority_baseline                     0.779000   
            config_A       majority_baseline                     0.779000   
            config_B       majority_baseline                     0.779000   
            config_A       random_forest                         0.782000   
                           xgboost                               0.785667   
            config_B       xgboost                               0.789333   
            combined       random_forest                         0.792667   
            config_A       logistic_regression                   0.789333   
            combined       xgboost                               0.789000   
                           logistic_regression                   0.789667   
            config_B       logistic_regression                   0.793000   
                           random_forest                         0.783000   
validated   combined       majority_baseline                     0.593087   
            config_A       majority_baseline                     0.593087   
            config_B       majority_baseline                     0.593087   
            config_A       xgboost                               0.612985   
            config_B       xgboost                               0.619010   
            combined       logistic_regression                   0.624988   
                           xgboost                               0.614666   
            config_A       logistic_regression                   0.624942   
            config_B       logistic_regression                   0.624521   
            config_A       random_forest                         0.582438   
            combined       ra


A positive gap means the model looked more accurate on the original, un-swapped order than it does once position is randomized -- i.e. part of its apparent accuracy came from position rather than code. A gap near zero across Config A is the result that supports using this model's static-feature accuracy at face value.
